# MeatGuard II — 01 · Comparison models and the no-distillation ablation

Trains the models MeatGuard-S is compared with, on the same session split and cropped images:

- **Lightweight baselines** — ImageNet-pretrained `mobilenet_v3_small`, `shufflenet_v2_x1_0`, `efficientnet_b0`,
  fine-tuned with the teacher recipe (`src/train_teacher.py`), saved under `runs/baselines/` so they are never
  candidates for the teacher.
- **MeatGuard-S without distillation** — the student recipe with cross-entropy only (`src/train_student.py distill
  --no-kd`), saved under `runs/student/scratch/`.

Independent of notebook 00; can run before or after it.

**Saved:** `best.pt`, `last.pt`, `history.csv`, `summary.json` in each run folder.

## 1. Setup
Upload the whole `MeatGuard_II` folder to Google Drive and the data zip (`crops/`, `masks/`, `sessions.csv`) next to
it, then edit the two paths below if yours differ. Results are written to `MeatGuard_II/runs/` on Drive, so they
survive disconnects; re-running a training cell resumes from the last saved epoch.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# --- paths: edit these -------------------------------------------------------------------------
REPO_DIR = "/content/drive/MyDrive/MeatGuard_II" if IN_COLAB else os.path.abspath("..")   # the codebase on Drive
DATA_SRC = "/content/drive/MyDrive/MeatGuard/full.zip" if IN_COLAB else "E:/Datasets/full"  # zip or folder: crops/, masks/, sessions.csv
# -------------------------------------------------------------------------------------------------
DATA_ROOT = "/content/data/full" if IN_COLAB else DATA_SRC     # local copy on the Colab VM (Drive is slow for many small files)
RUNS = os.path.join(REPO_DIR, "runs")
WORKERS = 4 if IN_COLAB else 0
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
if IN_COLAB:
    !pip -q install torch-pruning shap
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU visible"
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. Data

In [ ]:
# Copy the data to the VM once, then check it: every session in one partition, every listed image present
import zipfile, shutil
import pandas as pd
from pathlib import Path
if IN_COLAB and not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):
    if DATA_SRC.endswith(".zip"):
        with zipfile.ZipFile(DATA_SRC) as z:
            z.extractall(DATA_ROOT)
    else:
        shutil.copytree(DATA_SRC, DATA_ROOT)
if not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):      # zip made from the parent folder
    nested = [p.parent for p in Path(DATA_ROOT).glob("*/sessions.csv")]
    DATA_ROOT = str(nested[0]) if nested else DATA_ROOT
CROPS, SPLIT, MASKS = (os.path.join(DATA_ROOT, d) for d in ("crops", "sessions.csv", "masks"))

from src.train_teacher import check_split
table = pd.read_csv(SPLIT)
present = {p.relative_to(CROPS).with_suffix("").as_posix() for p in Path(CROPS).rglob("*.jpg")}
table["present"] = table["path"].str.replace(r"\.[^.]+$", "", regex=True).isin(present)
print("split counts:", check_split(SPLIT), "| masks/ present:", os.path.isdir(MASKS))
print(table.groupby(["split", "class"])["present"].agg(["sum", "count"]))
assert table["present"].all(), "Some images listed in sessions.csv are missing from crops/"

## 3. Lightweight baselines
`efficientnet_b0` is also a teacher candidate: if notebook 00 already trained it, that run is reused (same recipe and seed).

In [ ]:
BASELINES = ["mobilenet_v3_small", "shufflenet_v2_x1_0", "efficientnet_b0"]
BASE_RUNS = os.path.join(RUNS, "baselines")
for arch in BASELINES:
    done_as_teacher = os.path.join(RUNS, "teacher", arch, "summary.json")
    target = os.path.join(BASE_RUNS, arch)
    if os.path.exists(done_as_teacher) and not os.path.exists(os.path.join(target, "summary.json")):
        shutil.copytree(os.path.dirname(done_as_teacher), target, dirs_exist_ok=True)
        print("reused teacher run for", arch); continue
    print("=" * 30, arch, "=" * 30)
    !python -m src.train_teacher train --arch {arch} --data {CROPS} --split-file {SPLIT} --out {BASE_RUNS} \
        --num-workers {WORKERS}

## 4. MeatGuard-S without distillation (ablation)

In [ ]:
!python -m src.train_student distill --no-kd --data {CROPS} --split-file {SPLIT} --out {RUNS}/student \
    --num-workers {WORKERS}

In [ ]:
import matplotlib.pyplot as plt
def plot_curves(run_dirs, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for d in run_dirs:
        h = Path(d) / "history.csv"
        if not h.exists():
            continue
        df = pd.read_csv(h)
        for ax, col in zip(axes, ["train_loss", "val_macro_f1", "val_recall_spoiled"]):
            ax.plot(df["epoch"] + 1, df[col], label=Path(d).name)
    for ax, t in zip(axes, ["training loss", "validation macro-F1", "validation spoiled recall"]):
        ax.set_title(f"{title} {t}".strip()); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
    axes[1].legend(); plt.tight_layout(); plt.show()

plot_curves([os.path.join(BASE_RUNS, a) for a in BASELINES] + [os.path.join(RUNS, "student", "scratch")])

In [ ]:
# Bundle all result files (JSON/CSV, no weights) into one zip to share: runs/results_bundle.zip on Drive
!python -m src.collect_results --runs {RUNS} --out {RUNS}/results_bundle.zip